# 1 Why a pretext task produces a useful space

Lesson 1 handed you an encoder and showed that it dominates every other choice you can
make: the encoder moved accuracy by 10 to 40 points, the head architecture by less than
one. It never said where such an encoder comes from.

That is this lesson. And before the machinery — contrastive losses, EMA targets, masked
prediction — there is a prior question that the machinery only makes sense as an answer
to:

> **Why does training a model on a task you do not care about produce a representation
> that is good for tasks you do care about?**

This notebook answers it by building two spaces on the same images and measuring what
each one kept. Neither is the answer; together they show why a third thing is needed,
which is what the rest of the lesson builds.

## The instrument

Everything here is measured with a **linear probe**: freeze the encoder, fit a linear
classifier on top of its output, report accuracy. Nothing else is trained.

The probe is the right instrument precisely because it is weak. A deep head could
recover class structure from almost any representation, given enough capacity — and then
you would be measuring the head, not the encoder. A linear probe can only read structure
that is *already laid out linearly* in the space. It asks: did the encoder do the work,
or is it leaving the work to whatever comes next?

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from notebooktester import param
from torch import Tensor, nn
from torchvision.transforms import v2
from vectormesh.components import Serial

torch.manual_seed(42)

RES = 32       # 64 -> 32; this notebook is about direction, not absolute scores
DIM = 128      # width of every representation we compare
BATCH = 256
N_TRAIN, N_TEST = param(6_000, test=256), param(2_000, test=256)
EPOCHS = param(40, test=1)   # not a round number -- justified in 1.3

## 1.1 The data, split so that "generic" can be measured

EuroSAT again — the same aerial tiles as lesson 1, so the numbers here sit next to numbers
you have already produced.

The one deliberate choice: the ten classes are split into **five seen** and **five
unseen**. Everything that trains with labels only ever sees the first five. Every probe
is run on both halves. That split is what makes the word "generic" measurable rather than
rhetorical — a representation is generic if it is still useful for classes nobody
optimised it for.

In [ ]:
ds = load_dataset("blanchon/EuroSAT_RGB")
CLASSES = ds["train"].features["label"].names

SEEN = list(range(5))
UNSEEN = list(range(5, 10))
print("seen  :", [CLASSES[i] for i in SEEN])
print("unseen:", [CLASSES[i] for i in UNSEEN])

to_tensor = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Resize((RES, RES), antialias=True),
])


def pack(split, n, seed=0):
    d = split.shuffle(seed=seed)
    X = torch.stack([to_tensor(d[i]["image"].convert("RGB")) for i in range(n)])
    y = torch.tensor([d[i]["label"] for i in range(n)])
    return X, y


t0 = time.time()
Xtr, ytr = pack(ds["train"], N_TRAIN)
Xte, yte = pack(ds["test"], N_TEST)
print(f"packed {tuple(Xtr.shape)} train / {tuple(Xte.shape)} test in {time.time() - t0:.0f}s")

seen_tr = torch.isin(ytr, torch.tensor(SEEN))
seen_te = torch.isin(yte, torch.tensor(SEEN))
print(f"{int(seen_tr.sum())} seen-class train tiles, {int((~seen_tr).sum())} unseen-class")

## 1.2 The probe, and the floor it has to beat

Closed-form ridge regression onto one-hot targets — no optimiser, no learning rate, no
seed. Deterministic, a fraction of a second, and it cannot be accused of having been tuned
per representation, which matters when the whole notebook is a comparison.

Before any encoder exists, run it on the **raw pixels**. That is the floor. Any
representation that cannot beat 3072 raw numbers has not earned its existence.

In [ ]:
def probe(Ztr, ytr_, Zte, yte_, lam=1e-2):
    """Linear probe: closed-form ridge on one-hot targets. Returns accuracy."""
    Ztr = np.c_[Ztr, np.ones(len(Ztr))]
    Zte = np.c_[Zte, np.ones(len(Zte))]
    classes = np.unique(ytr_)
    Y = (ytr_[:, None] == classes[None, :]).astype(float)
    W = np.linalg.solve(Ztr.T @ Ztr + lam * np.eye(Ztr.shape[1]), Ztr.T @ Y)
    return float((classes[np.argmax(Zte @ W, 1)] == yte_).mean())


def evaluate(encode, name):
    """Probe a representation on the seen half and the unseen half separately."""
    Ztr, Zte = encode(Xtr), encode(Xte)
    ytr_np, yte_np = ytr.numpy(), yte.numpy()
    a = probe(Ztr[seen_tr.numpy()], ytr_np[seen_tr.numpy()], Zte[seen_te.numpy()], yte_np[seen_te.numpy()])
    b = probe(Ztr[~seen_tr.numpy()], ytr_np[~seen_tr.numpy()], Zte[~seen_te.numpy()], yte_np[~seen_te.numpy()])
    print(f"{name:28s} seen {a:.3f}   unseen {b:.3f}")
    return a, b


flatten = lambda X: X.reshape(len(X), -1).numpy()
results = {"raw pixels": evaluate(flatten, "raw pixels (the floor)")}

Chance is 0.200 on five classes. Raw pixels do better than chance, because aerial tiles of
water and forest genuinely differ in average colour — but this is the number to beat, and
it is beatable.

## 1.3 One encoder architecture, trained two different ways

Identical network in both experiments. Only the **training signal** differs, which is the
point: nothing below is an architecture result.

In [ ]:
class Encoder(nn.Module):
    """A small conv net. The lesson is in the training signal, not the architecture."""

    def __init__(self, in_ch: int = 3, dim: int = DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, dim),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


def encoder_fn(enc):
    """Frozen encoder -> numpy, in batches. Used by every probe below."""
    def f(X):
        enc.eval()
        with torch.no_grad():
            return torch.cat([enc(X[i:i + 512]) for i in range(0, len(X), 512)]).numpy()
    return f


def train(model, loss_fn, X, y=None, epochs=EPOCHS, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for _ in range(epochs):
        perm = torch.randperm(len(X))
        running = 0.0
        for i in range(0, len(X) - BATCH + 1, BATCH):
            idx = perm[i:i + BATCH]
            opt.zero_grad()
            loss = loss_fn(model, X[idx], None if y is None else y[idx])
            loss.backward()
            opt.step()
            running += loss.item()
        history.append(running / max(1, len(X) // BATCH))
    return history

### Space 1 — supervised, on the seen classes only

The obvious way to get a representation: train a classifier and throw the last layer
away. This is what "pretrained on ImageNet" means.

In [ ]:
torch.manual_seed(0)
sup_enc = Encoder()
sup = nn.Sequential(sup_enc, nn.Linear(DIM, len(SEEN)))

# remap the five seen labels to 0..4 for the cross-entropy head
remap = {c: i for i, c in enumerate(SEEN)}
Xs, ys = Xtr[seen_tr], torch.tensor([remap[int(v)] for v in ytr[seen_tr]])

hist_sup = train(sup, lambda m, xb, yb: F.cross_entropy(m(xb), yb), Xs, ys)
print(f"final supervised loss {hist_sup[-1]:.4f}")
results["supervised (seen labels)"] = evaluate(encoder_fn(sup_enc), "supervised (seen labels)")

A cross-entropy loss over five classes is an explicit instruction: *keep whatever separates
these five, discard everything else.* Discarding is not a side effect of that loss, it is
the objective. So the representation should be sharp on the seen classes and weaker on
classes nobody optimised it for.

It is. But **how much weaker depends entirely on how long you train**, and that turns out
to be the more useful lesson.

`EPOCHS` is 40 rather than a round number, and this is why. Measured on this machine,
3 seeds, probing the same encoder at checkpoints along one run:

| epochs | seen | unseen | gap |
|---|---|---|---|
| 4 | 0.8848 ± 0.0009 | 0.8873 ± 0.0021 | **-0.0025 ± 0.0024** |
| 8 | 0.9102 ± 0.0028 | 0.9036 ± 0.0080 | +0.0066 ± 0.0096 |
| 16 | 0.9256 ± 0.0021 | 0.9043 ± 0.0046 | +0.0212 ± 0.0066 |
| 24 | 0.9350 ± 0.0017 | 0.9094 ± 0.0080 | +0.0256 ± 0.0065 |
| 40 | 0.9455 ± 0.0040 | 0.9128 ± 0.0084 | +0.0326 ± 0.0124 |
| 60 | 0.9579 ± 0.0037 | 0.9088 ± 0.0010 | **+0.0491 ± 0.0044** |

Read the accuracy columns, not just the gap. **Seen accuracy climbs the whole way**, 0.885
to 0.958. **Unseen accuracy plateaus around 0.91 after epoch 24** and then stops. Past
that point, every additional epoch buys performance on the training classes and nothing at
all on the others — the encoder is not learning to see better, it is learning to see
*these five things* better. That is what specialisation looks like from the outside.

Now notice where a casual reading would have left you. At 8 epochs the gap is +0.007 with
a standard deviation of ±0.010: **you would conclude there is no gap, and you would be
wrong.** An earlier draft of this notebook did exactly that — trained for 12 epochs and
reported that supervised representations transfer for free. The epoch count changed the
sign of the conclusion.

This is why an epoch count in this course has to be justified by a curve rather than
chosen. It is not bookkeeping; it decides what you believe.

Now look at what this arm *cost*.

### Space 2 — an autoencoder, with no labels at all

The obvious fix: stop using labels. Ask the model to reproduce its own input through a
narrow bottleneck, and whatever survives the squeeze must be the important part.

The whole model is a `Serial` of two pieces — this is all an autoencoder is, and it is why
`5_unsupervised/src/autoencoder.py` no longer exists:

In [ ]:
class Decoder(nn.Module):
    """DIM -> an image again. Deliberately plain: the encoder is what we are studying."""

    def __init__(self, dim: int = DIM, out_ch: int = 3, res: int = RES):
        super().__init__()
        self.res = res
        self.net = nn.Sequential(
            nn.Linear(dim, 256), nn.GELU(),
            nn.Linear(256, out_ch * res * res), nn.Sigmoid(),
        )
        self.out_ch = out_ch

    def forward(self, z: Tensor) -> Tensor:
        return self.net(z).reshape(-1, self.out_ch, self.res, self.res)


torch.manual_seed(0)
ae_enc = Encoder()
autoencoder = Serial([ae_enc, Decoder()])

hist_ae = train(autoencoder, lambda m, xb, _: F.mse_loss(m(xb), xb), Xtr)
print(f"final reconstruction MSE {hist_ae[-1]:.5f}")
results["autoencoder (no labels)"] = evaluate(encoder_fn(ae_enc), "autoencoder (no labels)")

Look at what it reconstructs before looking at what it scores.

In [ ]:
autoencoder.eval()
with torch.no_grad():
    recon = autoencoder(Xte[:6])

fig, axes = plt.subplots(2, 6, figsize=(12, 4.2))
for j in range(6):
    axes[0, j].imshow(Xte[j].permute(1, 2, 0)); axes[0, j].set_title(CLASSES[yte[j]], fontsize=8)
    axes[1, j].imshow(recon[j].permute(1, 2, 0))
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].set_ylabel("input"); axes[1, 0].set_ylabel("reconstruction")
plt.suptitle("what the autoencoder chose to keep")
plt.tight_layout(); plt.show()

The reconstructions are blurry colour-and-texture summaries. That is not a bug to be
trained away — it is the MSE loss doing exactly what it was asked. Squared error over
pixels is dominated by *large, bright, low-frequency* regions, so the cheapest way to cut
it is to get the average colour and coarse texture right. Fine structure contributes
little to the loss and gets spent last, if at all.

So the autoencoder's bottleneck is full of "this tile is greenish and smooth". Some of
that correlates with class. Much of it is brightness and haze that no label cares about —
and it still lands within a few points of the supervised encoder, having never been shown
a single label.

## 1.4 The table, and the actual point of the lesson

In [ ]:
print(f"{'representation':28s} {'seen':>7s} {'unseen':>8s}   gap")
print("-" * 56)
for name, (a, b) in results.items():
    print(f"{name:28s} {a:7.3f} {b:8.3f}   {a - b:+.3f}")

Two limitations, opposite in kind, and one column the table does not show:

| representation | labels used | unseen accuracy |
|---|---|---|
| raw pixels | 0 | ~0.42 |
| supervised | ~3,100 | ~0.91, and plateaued |
| autoencoder | **0** | ~0.86 |

- **Supervised is sharp but increasingly parochial**, and it costs labels. Every epoch past
  ~24 sharpens it on the five classes it was paid for and does nothing for the rest.
- **The autoencoder is generic but blunt.** It never saw a label, so it has no reason to
  prefer one class boundary over another — and no reason to discard anything either. It
  spends its bottleneck on whatever reduces squared error, which is largely brightness.

Neither is what lesson 1's encoder was. That one was generic **and** sharp, and it cost no
labels on your data at all. So the question sharpens into something concrete:

> Is there a training signal that needs no labels, like reconstruction, but that still
> tells the model **what to throw away**, like a supervised loss does?

Note how close the free arm already is — ~0.86 against ~0.91, at a cost of zero labels
versus three thousand. Those five points are what the rest of this lesson is about.

In [ ]:
def knn_purity(Z, y, k=5):
    """Fraction of a point's k nearest neighbours that share its label."""
    Zt = torch.tensor(Z, dtype=torch.float32)
    Zt = F.normalize(Zt, dim=-1)
    sim = Zt @ Zt.T
    sim.fill_diagonal_(-torch.inf)
    nn_idx = sim.topk(k, dim=-1).indices
    return float((y[nn_idx] == y[:, None]).float().mean())


print(f"{'representation':28s} 5-NN label purity")
print("-" * 50)
for name, enc in [("raw pixels", flatten),
                  ("supervised (seen labels)", encoder_fn(sup_enc)),
                  ("autoencoder (no labels)", encoder_fn(ae_enc))]:
    print(f"{name:28s} {knn_purity(enc(Xte), yte):.3f}")

Purity is what "organised by" means, made countable, and it diagnoses the gap.

The supervised space puts same-label tiles next to each other because that is precisely
what it was paid to do. The autoencoder's space puts **similar-looking** tiles next to
each other — which overlaps with class only as far as appearance and category happen to
coincide. Its purity sits well below the supervised space's while its probe accuracy sits
only a few points below, and both facts have the same cause: the bottleneck is spending
capacity on brightness and haze that no label cares about, and a linear probe can still
partly read around that, but a nearest neighbour cannot.

The remaining gap is therefore not a capacity problem. It is that **nothing ever told the
autoencoder what to ignore.**

**This is the sentence to carry out of this notebook:**

> The pretext task is not a trick for getting labels for free. It is the choice of **what
> the representation is allowed to forget.**

Reconstruction is told to forget *nothing*, so it keeps brightness. Classification is told to
forget everything except five boundaries, obeys that more literally the longer it trains,
and pays for the instruction in labels. The next notebook takes the
third option, which needs no labels and still gives an instruction: *state the invariances
yourself*. Two augmented views of one tile must land in the same
place — so whatever your augmentation changes is, by construction, what the space is told
to discard. Rotate the tile and the label does not change, so rotation should not move the
vector.

That is a design decision you make, and get wrong, and measure. Which is exactly what
`02_selfsupervised` does next — including what happens when the augmentation destroys the
very thing the task depended on.

## Exercises

**E1 — reproduce the table in 1.3, then break it.** Re-run the epoch sweep at one seed and
confirm the gap grows. Then change the *domain* rather than the schedule: probe the
supervised encoder on the DTD texture cache from lesson 1 instead of on held-out EuroSAT
classes. EuroSAT's ten classes are all aerial land-cover, so the held-out half is not
really a different *kind* of thing. How much wider does the gap get when it is?

**E2 — is the bottleneck the reason?** `DIM` is 128. Re-run the autoencoder at 16 and at
512 and probe each. Does a tighter squeeze force it to keep more class-relevant structure,
as the folklore says, or does it just lose information? Report the number, not the
folklore.

**E3 — a denoising autoencoder.** Feed the autoencoder a noisy tile and ask for the clean
one back. This is a pretext task with an invariance stated in it — "noise should not
matter". Does probe accuracy on the unseen classes move? This is the smallest possible
version of the idea the whole rest of the lesson is built on.

**E4 — the label-budget curve.** The supervised arm used ~3,100 labels, the autoencoder
none. Retrain the supervised encoder on 50, 200, 1000 and all labels and plot unseen-class
probe accuracy against label count, with the autoencoder as a flat horizontal line. Below
how many labels does the *unsupervised* representation win?

**E6 — when does the plateau start?** 1.3 shows unseen accuracy flattening near epoch 24
while seen accuracy keeps climbing. Find that elbow — then ask the uncomfortable question:
with no held-out classes to probe, which is the normal situation, what signal available
*during training* would have told you the encoder had stopped generalising and started
specialising?

**E5 — the floor is not always low.** On which of the two halves do raw pixels do best, and
what does that tell you about how much of EuroSAT is solvable by average colour alone?